# 🔍 Introduction to Retrieval - The Heart of RAG

## What is Retrieval?

**Retrieval is the process of finding relevant information from a large collection of documents based on a query.**

### The RAG Pipeline:

```
User Query
    ↓
🔍 RETRIEVAL ← We are here!
    ↓
Relevant Documents
    ↓
LLM Generation
    ↓
Answer
```

## Why Retrieval Matters for RAG 🎯

**The quality of retrieval directly determines:**
1. **Answer accuracy** - Right context = right answer
2. **Factual grounding** - Real documents = less hallucination
3. **Response relevance** - Better retrieval = better responses
4. **System performance** - Fast retrieval = fast responses

### The Golden Rule:
```
Garbage In (bad retrieval) → Garbage Out (bad answers)
Gold In (good retrieval) → Gold Out (great answers) ✨
```

## Types of Retrieval Methods

### 1. **Sparse Retrieval** (Traditional) 📊
- **How**: Keyword/term matching (TF-IDF, BM25)
- **Pros**: Fast, interpretable, no ML needed
- **Cons**: Vocabulary mismatch, no semantic understanding
- **Example**: "apple fruit" won't match "healthy apples"

### 2. **Dense Retrieval** (Modern) 🧠
- **How**: Semantic embeddings + vector similarity
- **Pros**: Understands meaning, handles synonyms
- **Cons**: Requires GPU, less interpretable
- **Example**: "apple fruit" WILL match "healthy apples"

### 3. **Hybrid Retrieval** (Best of Both) 🚀
- **How**: Combines sparse + dense
- **Pros**: Best accuracy, robust
- **Cons**: More complex
- **Example**: Used in production by Google, Bing

## Key Concepts

### Embeddings
```python
Text: "The quick brown fox"
    ↓ Embedding Model
Vector: [0.23, -0.45, 0.12, ...] (768 dimensions)
```

### Similarity Metrics
- **Cosine Similarity**: Most common (angle between vectors)
- **Dot Product**: Fast, used in many systems
- **Euclidean Distance**: Geometric distance

### Vector Database
```
Documents → Embeddings → Vector DB
                              ↓
Query → Embedding → Similarity Search → Top-K Results
```

## 1. Simple Retrieval Example

In [1]:
# Simple keyword-based retrieval
documents = [
    "Python is a programming language used for web development and data science.",
    "Machine learning is a subset of artificial intelligence.",
    "Natural language processing helps computers understand human language.",
    "Deep learning uses neural networks with multiple layers.",
    "RAG combines retrieval with generation for better AI responses."
]

def simple_retrieval(query, documents, top_k=3):
    """Naive keyword matching"""
    query_words = set(query.lower().split())
    scores = []
    
    for doc in documents:
        doc_words = set(doc.lower().split())
        # Count matching words
        overlap = len(query_words & doc_words)
        scores.append(overlap)
    
    # Get top-k indices
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    
    return [(documents[i], scores[i]) for i in top_indices]

# Test
query = "machine learning and AI"
results = simple_retrieval(query, documents)

print(f"Query: {query}\n")
print("Top Results:")
for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. Score {score}: {doc}")

print("\n💡 Simple but works! However, misses semantic similarity.")

Query: machine learning and AI

Top Results:
1. Score 2: Machine learning is a subset of artificial intelligence.
2. Score 1: Python is a programming language used for web development and data science.
3. Score 1: Deep learning uses neural networks with multiple layers.

💡 Simple but works! However, misses semantic similarity.


## 2. Understanding the Retrieval Problem

In [2]:
# The semantic gap problem
query1 = "programming language"
query2 = "coding language"  # Different words, same meaning!

results1 = simple_retrieval(query1, documents)
results2 = simple_retrieval(query2, documents)

print("Query 1: 'programming language'")
print(f"Best match: {results1[0][0][:50]}... (score: {results1[0][1]})\n")

print("Query 2: 'coding language'")
print(f"Best match: {results2[0][0][:50]}... (score: {results2[0][1]})\n")

print("❌ Problem: Same meaning, different results!")
print("✅ Solution: Semantic retrieval with embeddings")

Query 1: 'programming language'
Best match: Python is a programming language used for web deve... (score: 2)

Query 2: 'coding language'
Best match: Python is a programming language used for web deve... (score: 1)

❌ Problem: Same meaning, different results!
✅ Solution: Semantic retrieval with embeddings


## 3. Embedding-based Retrieval (Preview)

In [3]:
# Install sentence-transformers
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_retrieval(query, documents, top_k=3):
    """Retrieval using semantic embeddings"""
    # Encode query and documents
    query_embedding = model.encode(query)
    doc_embeddings = model.encode(documents)
    
    # Calculate cosine similarity
    similarities = np.dot(doc_embeddings, query_embedding) / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    
    # Get top-k
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    return [(documents[i], similarities[i]) for i in top_indices]

# Test with both queries
print("SEMANTIC RETRIEVAL:\n")

for q in [query1, query2]:
    results = semantic_retrieval(q, documents)
    print(f"Query: '{q}'")
    print(f"Best match: {results[0][0][:50]}... (similarity: {results[0][1]:.3f})\n")

print("✅ Both queries return the SAME result - semantic understanding!")

SEMANTIC RETRIEVAL:

Query: 'programming language'
Best match: Python is a programming language used for web deve... (similarity: 0.545)

Query: 'coding language'
Best match: Natural language processing helps computers unders... (similarity: 0.406)

✅ Both queries return the SAME result - semantic understanding!


## 4. Retrieval Metrics

In [4]:
# Understanding retrieval quality metrics

# Ground truth: which documents are actually relevant
query = "What is machine learning?"
relevant_docs = {1, 3}  # Indices of relevant documents

# Retrieved documents (indices)
retrieved_docs = {1, 2, 4}

# Calculate metrics
true_positives = len(relevant_docs & retrieved_docs)
precision = true_positives / len(retrieved_docs)
recall = true_positives / len(relevant_docs)
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Retrieval Metrics:\n")
print(f"Query: {query}")
print(f"Relevant docs: {relevant_docs}")
print(f"Retrieved docs: {retrieved_docs}")
print(f"\nPrecision: {precision:.2f} (how many retrieved are relevant?)")
print(f"Recall: {recall:.2f} (how many relevant were retrieved?)")
print(f"F1-Score: {f1:.2f} (harmonic mean)")

print("\n💡 Higher is better for all metrics!")

Retrieval Metrics:

Query: What is machine learning?
Relevant docs: {1, 3}
Retrieved docs: {1, 2, 4}

Precision: 0.33 (how many retrieved are relevant?)
Recall: 0.50 (how many relevant were retrieved?)
F1-Score: 0.40 (harmonic mean)

💡 Higher is better for all metrics!


## 5. Retrieval Pipeline Overview

In [5]:
# Complete RAG retrieval pipeline

class SimpleRAGRetriever:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None
    
    def index_documents(self, documents):
        """Index documents by creating embeddings"""
        self.documents = documents
        print(f"Indexing {len(documents)} documents...")
        self.embeddings = self.model.encode(documents, show_progress_bar=True)
        print("✅ Indexing complete!")
    
    def retrieve(self, query, top_k=3):
        """Retrieve top-k most similar documents"""
        query_embedding = self.model.encode(query)
        
        # Cosine similarity
        similarities = np.dot(self.embeddings, query_embedding) / (
            np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        return [
            {
                'document': self.documents[i],
                'score': float(similarities[i]),
                'rank': rank + 1
            }
            for rank, i in enumerate(top_indices)
        ]

# Test the retriever
retriever = SimpleRAGRetriever()
retriever.index_documents(documents)

query = "How does AI understand language?"
results = retriever.retrieve(query, top_k=3)

print(f"\nQuery: {query}\n")
print("Retrieved Documents:")
for result in results:
    print(f"\nRank {result['rank']} (Score: {result['score']:.3f}):")
    print(f"  {result['document']}")

Indexing 5 documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexing complete!

Query: How does AI understand language?

Retrieved Documents:

Rank 1 (Score: 0.571):
  Natural language processing helps computers understand human language.

Rank 2 (Score: 0.394):
  Machine learning is a subset of artificial intelligence.

Rank 3 (Score: 0.390):
  RAG combines retrieval with generation for better AI responses.


## 6. Comparison: Sparse vs Dense

In [6]:
# Compare retrieval methods
test_queries = [
    "programming languages",
    "AI and machine learning",
    "understanding text",
    "neural networks"
]

print("Sparse vs Dense Retrieval Comparison:\n")
print(f"{'Query':<25} {'Sparse (keywords)':<35} {'Dense (semantic)'}")
print("="*95)

for query in test_queries:
    # Sparse retrieval
    sparse_result = simple_retrieval(query, documents, top_k=1)[0]
    
    # Dense retrieval
    dense_result = semantic_retrieval(query, documents, top_k=1)[0]
    
    print(f"{query:<25} {sparse_result[0][:33]:<35} {dense_result[0][:40]}")

print("\n💡 Dense retrieval often finds better semantic matches!")

Sparse vs Dense Retrieval Comparison:

Query                     Sparse (keywords)                   Dense (semantic)
programming languages     Python is a programming language    Python is a programming language used fo
AI and machine learning   Machine learning is a subset of a   Machine learning is a subset of artifici
understanding text        Python is a programming language    Natural language processing helps comput
neural networks           Deep learning uses neural network   Deep learning uses neural networks with 

💡 Dense retrieval often finds better semantic matches!


## 7. Real-World RAG Example

In [7]:
# Realistic RAG knowledge base
knowledge_base = [
    "RAG (Retrieval-Augmented Generation) combines information retrieval with text generation.",
    "The retrieval component finds relevant documents from a knowledge base using embeddings.",
    "Retrieved documents provide context to the language model for generating accurate responses.",
    "Vector databases like Pinecone, Weaviate, and Chroma store document embeddings efficiently.",
    "Embedding models like BERT, Sentence-BERT, and E5 convert text into dense vectors.",
    "Cosine similarity is commonly used to find similar documents in vector space.",
    "RAG systems reduce hallucination by grounding responses in retrieved facts.",
    "Hybrid retrieval combines keyword search (BM25) with semantic search for better accuracy.",
]

# Create retriever
rag_retriever = SimpleRAGRetriever()
rag_retriever.index_documents(knowledge_base)

# User questions
questions = [
    "What is RAG?",
    "How does retrieval work in RAG?",
    "What are vector databases?",
    "How to reduce hallucination?"
]

print("RAG Question Answering (Retrieval Only):\n")
print("="*80)

for question in questions:
    results = rag_retriever.retrieve(question, top_k=2)
    print(f"\n❓ Question: {question}")
    print(f"📄 Top Retrieved Context:")
    for result in results:
        print(f"   [{result['score']:.3f}] {result['document']}")

print("\n💡 These contexts would be passed to an LLM for answer generation!")

Indexing 8 documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexing complete!
RAG Question Answering (Retrieval Only):


❓ Question: What is RAG?
📄 Top Retrieved Context:
   [0.487] RAG systems reduce hallucination by grounding responses in retrieved facts.
   [0.479] RAG (Retrieval-Augmented Generation) combines information retrieval with text generation.

❓ Question: How does retrieval work in RAG?
📄 Top Retrieved Context:
   [0.515] RAG (Retrieval-Augmented Generation) combines information retrieval with text generation.
   [0.498] RAG systems reduce hallucination by grounding responses in retrieved facts.

❓ Question: What are vector databases?
📄 Top Retrieved Context:
   [0.551] Vector databases like Pinecone, Weaviate, and Chroma store document embeddings efficiently.
   [0.422] Cosine similarity is commonly used to find similar documents in vector space.

❓ Question: How to reduce hallucination?
📄 Top Retrieved Context:
   [0.596] RAG systems reduce hallucination by grounding responses in retrieved facts.
   [-0.020] Hybrid retrieval 

## Key Takeaways 🎯

### ✅ Core Concepts:

1. **Retrieval is crucial** - It determines RAG quality
2. **Two main types**:
   - Sparse: Keyword matching (BM25, TF-IDF)
   - Dense: Semantic embeddings (Neural)
3. **Dense > Sparse** for semantic understanding
4. **Hybrid = Best** in production

### 🔑 Retrieval Pipeline:

```python
# 1. Index documents
embeddings = model.encode(documents)

# 2. Encode query
query_embedding = model.encode(query)

# 3. Find similar
similarities = cosine_similarity(query_embedding, embeddings)

# 4. Return top-k
top_docs = get_top_k(similarities)
```

### 📊 Key Metrics:

- **Precision**: % of retrieved docs that are relevant
- **Recall**: % of relevant docs that were retrieved
- **F1-Score**: Harmonic mean of precision and recall
- **MRR**: Mean Reciprocal Rank (position of first relevant)
- **NDCG**: Normalized Discounted Cumulative Gain

### 🚀 For Production RAG:

**Must-haves:**
1. Dense retrieval with embeddings
2. Efficient vector database
3. Fast similarity search (ANN)
4. Good embedding model
5. Proper chunking strategy

**Nice-to-haves:**
1. Hybrid retrieval (sparse + dense)
2. Re-ranking step
3. Query expansion
4. Metadata filtering
5. Multi-stage retrieval

### 💡 Common Patterns:

```python
# Pattern 1: Simple RAG
docs = retrieve(query, top_k=5)
context = '\n'.join(docs)
answer = llm.generate(context, query)

# Pattern 2: RAG with re-ranking
candidates = retrieve(query, top_k=100)  # Over-retrieve
docs = rerank(candidates, query, top_k=5)  # Re-rank
answer = llm.generate(docs, query)

# Pattern 3: Multi-hop RAG
docs1 = retrieve(query)
sub_queries = extract_sub_queries(docs1)
docs2 = retrieve(sub_queries)
answer = llm.generate(docs1 + docs2, query)
```

### ⚠️ Common Pitfalls:

1. **Poor chunking** → Bad retrieval
2. **Wrong embedding model** → Semantic mismatch
3. **Too few documents** → Missing context
4. **Too many documents** → Noisy context
5. **No re-ranking** → Suboptimal ordering

### 📈 Optimization Tips:

1. **Chunk size**: 256-512 tokens optimal
2. **Overlap**: 1-2 sentences between chunks
3. **Top-k**: Start with 3-5, tune based on LLM context
4. **Embedding model**: Choose based on domain
5. **Similarity metric**: Cosine for normalized, dot product for speed

## What's Next? 🔜

In the following notebooks, you'll learn:

1. **`01_Dense_Retrieval.ipynb`** - Deep dive into embeddings
2. **`02_Sparse_Retrieval.ipynb`** - BM25, TF-IDF techniques
3. **`03_Hybrid_Retrieval.ipynb`** - Combining the best of both
4. **`04_Vector_Databases.ipynb`** - Pinecone, Weaviate, Chroma
5. **`05_Similarity_Metrics.ipynb`** - Cosine, dot product, distance
6. **`06_Retrieval_Optimization.ipynb`** - Production techniques

Ready to master retrieval? Let's go! 🚀